# Riding the Fixed-Point Curve · companion notebook: MNIST and CIFAR-10

Companion to **"Riding the Fixed-Point Curve: Image Classification in the Continuous Thought Machine"**
(T. Hannagan, NeurReps 2026, Extended Abstract Track, non-archival; extended version on arXiv:
[arxiv.org/abs/2610.19220](https://arxiv.org/abs/2610.19220)). The main
notebook, `ctm_reproduce_imagenet_1000cls`, reproduces every ImageNet figure, table and number. **This one reproduces
§8 (Smaller models) of the extended (arXiv) version: the MNIST and CIFAR rows of Table 2 and every number in the
section**, and nothing else. §8 has no figure.

Two models are trained (or loaded from checkpoint), three seeds each: a $D{=}128$ MNIST CTM and a $D{=}128$
CIFAR-10 CTM with a ResNet-9 backbone. Both use a dense synchronization matrix (every tracked entry is a pair
among the first 16 neurons, 136 entries per slice), unlike the ImageNet checkpoint's random pairing.

### What this notebook reproduces

Every analysis cell carries a tag such as **[8.2]** in its heading (paper section, then position).

**Table 2 (MNIST and CIFAR rows)**

| Column | Cell | Value in the paper (MNIST / CIFAR) |
|---|---|---|
| acc. | Part 2 · *Train or load the two models* (printed there; collected in [8.6]) | 0.95 / 0.84 |
| $\rho(J)$ (median) | [8.1] | 0.10 / 0.068 |
| $\sigma_{\max}(J)$ (median / max) | [8.1] | 0.49 / 0.77 and 0.36 / 3.06 |
| attn. ($H/\ln L$) | [8.3] | 0.97 / 0.90 ($L$ = 49 / 64 patches) |
| frozen, shuffled | [8.3] | +0% / +0% and +0.3% / +0.2% |
| all columns, collected | [8.6] | |

**Numbers quoted in the text of §8**

| Quantity | Cell |
|---|---|
| CIFAR with the original backbone: 0.64 (with ResNet-9: 0.84) | Part 2, after setting `CIFAR_BACKBONE = 'official'` and `CIFAR_STEPS = 6000` in the configuration |
| $\rho(J)$ near a tenth; $\sigma_{\max}<1$ throughout on MNIST, in the median on CIFAR | [8.1] |
| normal share $\sum\lvert\lambda\rvert^2/\lVert J\rVert_F^2$: ≈ 0.29 at initialization (Ginibre 0.50), 0.16 / 0.095 after training | [8.2] |
| attention ≈ 44 of 49 patches (MNIST), ≈ 42 of 64 (CIFAR); frozen/swapped query moves accuracy by at most a few tenths of a percent; only removing or flattening attention hurts | [8.3] |
| $\cos(\Delta z^t,-\nabla_z H)$: peak $+0.69$ at tick 1, crossing at tick 5 (CIFAR); peak $+0.65$ at tick 9, crossing at tick 17 (MNIST) | [8.4] |
| retention of the earliest contributions at the decision tick: median 0.90 (MNIST) / 0.63 (CIFAR) | [8.5] |

**Correction to the accepted version.** Its Table 3 gives `attn.` = 1.00 for both small models. That value
normalised the entropy by an estimate of the token count taken from the entropy itself, which biases it towards 1.
[8.3] reads $L$ from the attention tensor: MNIST attention is close to uniform (0.97), CIFAR attention is
moderately selective (0.90). The conclusion is unchanged: on both, the query plays no role.

Run top to bottom. With checkpoints present on Drive, models load almost instantly; training everything from
scratch (three seeds of the real, shuffled-label and untrained models) takes about two hours on a Colab GPU,
mostly CIFAR.

**How to cite.**

```bibtex
@misc{hannagan2026riding,
  title         = {Riding the Fixed-Point Curve: Image Classification in the Continuous Thought Machine},
  author        = {Hannagan, Thomas},
  year          = {2026},
  eprint        = {2610.19220},
  archivePrefix = {arXiv},
  primaryClass  = {cs.LG},
  note          = {NeurIPS 2026 Workshop on Symmetry and Geometry in Neural Representations (NeurReps)}
}
```


# Part 1 · Setup

### Configuration

What to run, seeds, and paths.

In [ ]:
# ---- what to run -------------------------------------------------------------------------------
N_SEEDS       = 3          # the manuscript reports mean +- sd across 3 seeds
FORCE_RETRAIN = False      # True ignores the checkpoint cache and retrains everything

# ---- checkpoint storage -------------------------------------------------------------------------
# Trained models are cached so that a rerun does analysis only. On Colab the local disk is wiped
# whenever the runtime restarts, so point the cache at Drive and the models survive. Anywhere else
# (or if the mount fails) this falls back to ./ckpts automatically, so the notebook remains
# self-contained for anyone reproducing the paper without access to this Drive.
# DRIVE_DIR is a MOUNTED PATH, not a drive.google.com URL. A folder you own lives under MyDrive/;
# a folder shared with you lives under 'Shared with me', which the mount does NOT expose -- in that
# case open it in Drive and use "Add shortcut to Drive" so it appears under MyDrive/ too.
# On mount, the cell below prints what is actually in MyDrive/, so the right name is easy to copy.
USE_DRIVE  = True
DRIVE_DIR  = '/content/drive/MyDrive/ctm_project'   # <-- set to your folder name
LOCAL_DIR  = '.'                                     # fallback when Drive is unavailable

# ---- conventions (§2: t_dec is the most certain tick) ------------------------------------------
DECISION_MODE   = 'most_certain_tick'   # t_dec = argmax_t certainty, the CTM authors' own convention
TRUNCATE_WINDOW = True                  # every t_dec-dependent quantity is computed over [1, t_dec]

# ---- CIFAR-10 backbone (§8: ResNet-9; 'official' gives the 0.64 quoted for the original backbone) -
# 'official' reproduces the paper. 'resnet9' swaps in a stronger perception stack to test whether the
# MNIST/CIFAR asymmetries in the paper track task difficulty or merely how well this small model has
# solved the task. D, T and n_synch are held fixed either way, so every number stays comparable --
# only the backbone, the augmentation, and the CIFAR training schedule change.
CIFAR_BACKBONE = 'resnet9'     # 'official' | 'resnet9'  <-- resnet9: strong CIFAR for the paper's 'performs well' claim
CIFAR_STEPS    = 6000            # CIFAR-10 training steps. 6000 with the 'official' backbone is what
                                 # the manuscript reports. The 'resnet9' backbone underfits at 6000 and
                                 # was run at 20000; intermediate values vary accuracy at FIXED
                                 # architecture, which is the clean way to ask whether a result tracks
                                 # accuracy or tracks capacity.
CIFAR_AUGMENT  = (CIFAR_BACKBONE == 'resnet9')   # random crop + h-flip; standard for CIFAR-10

# ---- evaluation sample sizes -------------------------------------------------------------------
N_GRAD  = 64               # [8.4] images per seed for the entropy-gradient alignment

print(f'N_SEEDS={N_SEEDS} FORCE_RETRAIN={FORCE_RETRAIN} DECISION_MODE={DECISION_MODE!r}')
print(f'USE_DRIVE={USE_DRIVE}  DRIVE_DIR={DRIVE_DIR!r}')
print(f'CIFAR_BACKBONE={CIFAR_BACKBONE!r}  CIFAR_STEPS={CIFAR_STEPS}  CIFAR_AUGMENT={CIFAR_AUGMENT}')

### Model source

The official CTM implementation, pinned.

In [ ]:
import math, random, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable a GPU (Runtime -> Change runtime type)'
print('Device:', torch.cuda.get_device_name(0))

import math, random, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable a GPU (Runtime -> Change runtime type)'
print('Device:', torch.cuda.get_device_name(0))

class Identity(nn.Module):
    """Identity Module."""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x


class Squeeze(nn.Module):
    """Squeeze Module."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        return x.squeeze(self.dim)


class SuperLinear(nn.Module):
    """SuperLinear Layer: Implements Neuron-Level Models (NLMs) for the CTM."""
    def __init__(self, in_dims, out_dims, N, dropout=0.0):
        super().__init__()
        self.in_dims = in_dims
        self.dropout = nn.Dropout(dropout) if dropout > 0 else Identity()
        self.register_parameter('w1', nn.Parameter(
            torch.empty((in_dims, out_dims, N)).uniform_(
                -1/math.sqrt(in_dims + out_dims),
                 1/math.sqrt(in_dims + out_dims)
            ), requires_grad=True)
        )
        self.register_parameter('b1', nn.Parameter(torch.zeros((1, N, out_dims)), requires_grad=True))

    def forward(self, x):
            out = self.dropout(x)
            out = torch.einsum('BDM,MHD->BDH', out, self.w1) + self.b1
            out = out.squeeze(-1)
            return out


def compute_normalized_entropy(logits, reduction='mean'):
    """Computes the normalized entropy for certainty-loss."""
    preds = F.softmax(logits, dim=-1)
    log_preds = torch.log_softmax(logits, dim=-1)
    entropy = -torch.sum(preds * log_preds, dim=-1)
    num_classes = preds.shape[-1]
    max_entropy = torch.log(torch.tensor(num_classes, dtype=torch.float32))
    normalized_entropy = entropy / max_entropy
    if len(logits.shape) > 2 and reduction == 'mean':
        normalized_entropy = normalized_entropy.flatten(1).mean(-1)
    return normalized_entropy


class ContinuousThoughtMachine(nn.Module):
    def __init__(self,
                 iterations,
                 d_model,
                 d_input,
                 memory_length,
                 heads,
                 n_synch_out,
                 n_synch_action,
                 out_dims,
                 memory_hidden_dims,
                 dropout=0.0
                 ):
        super(ContinuousThoughtMachine, self).__init__()

        # --- Core Parameters ---
        self.iterations = iterations
        self.d_model = d_model
        self.d_input = d_input
        self.memory_length = memory_length
        self.n_synch_out = n_synch_out
        self.n_synch_action = n_synch_action
        self.out_dims = out_dims
        self.memory_length = memory_length
        self.memory_hidden_dims = memory_hidden_dims

        # --- Input Processing  ---
        self.backbone = nn.Sequential(
            nn.LazyConv2d(d_input, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(d_input),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.LazyConv2d(d_input, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(d_input),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.attention = nn.MultiheadAttention(self.d_input, heads, dropout, batch_first=True)
        self.kv_proj = nn.Sequential(nn.LazyLinear(self.d_input), nn.LayerNorm(self.d_input))
        self.q_proj = nn.LazyLinear(self.d_input)

        # --- Core CTM Modules ---
        self.synapses = nn.Sequential(
                nn.Dropout(dropout),
                nn.LazyLinear(d_model * 2),
                nn.GLU(),
                nn.LayerNorm(d_model)
            )
        self.trace_processor = nn.Sequential(
            SuperLinear(in_dims=memory_length, out_dims=2 * memory_hidden_dims, N=d_model, dropout=dropout),
            nn.GLU(),
            SuperLinear(in_dims=memory_hidden_dims, out_dims=2, N=d_model, dropout=dropout),
            nn.GLU(),
            Squeeze(-1)
        )

        #  --- Start States ---
        self.register_parameter('start_activated_state', nn.Parameter(
                torch.zeros((d_model)).uniform_(-math.sqrt(1/(d_model)), math.sqrt(1/(d_model))),
                requires_grad=True
            ))

        self.register_parameter('start_trace', nn.Parameter(
            torch.zeros((d_model, memory_length)).uniform_(-math.sqrt(1/(d_model+memory_length)), math.sqrt(1/(d_model+memory_length))),
            requires_grad=True
        ))

        # --- Synchronisation ---
        self.synch_representation_size_action = (self.n_synch_action * (self.n_synch_action+1))//2
        self.synch_representation_size_out = (self.n_synch_out * (self.n_synch_out+1))//2

        for synch_type, size in [('action', self.synch_representation_size_action), ('out', self.synch_representation_size_out)]:
            print(f"Synch representation size {synch_type}: {size}")

        self.set_synchronisation_parameters('out')
        self.set_synchronisation_parameters('action')

        # --- Output Procesing ---
        self.output_projector = nn.Sequential(nn.LazyLinear(self.out_dims))

    def set_synchronisation_parameters(self, synch_type: str):
        synch_representation_size = self.synch_representation_size_action if synch_type == 'action' else self.synch_representation_size_out
        self.register_parameter(f'decay_params_{synch_type}', nn.Parameter(torch.zeros(synch_representation_size), requires_grad=True))

    def compute_synchronisation(self, activated_state, decay_alpha, decay_beta, r, synch_type):
        if synch_type == 'action':
            n_synch = self.n_synch_action
            selected_left = selected_right = activated_state[:, -n_synch:]
        elif synch_type == 'out':
            n_synch = self.n_synch_out
            selected_left = selected_right = activated_state[:, :n_synch]

        outer = selected_left.unsqueeze(2) * selected_right.unsqueeze(1)
        i, j = torch.triu_indices(n_synch, n_synch)
        pairwise_product = outer[:, i, j]

        if decay_alpha is None or decay_beta is None:
            decay_alpha = pairwise_product
            decay_beta = torch.ones_like(pairwise_product)
        else:
            decay_alpha = r * decay_alpha + pairwise_product
            decay_beta = r * decay_beta + 1

        synchronisation = decay_alpha / (torch.sqrt(decay_beta))
        return synchronisation, decay_alpha, decay_beta

    def compute_features(self, x):
        input_features = self.backbone(x)
        kv = self.kv_proj(input_features.flatten(2).transpose(1, 2))
        return kv

    def compute_certainty(self, current_prediction):
        ne = compute_normalized_entropy(current_prediction)
        current_certainty = torch.stack((ne, 1-ne), -1)
        return current_certainty

    def forward(self, x, track=False):
        B = x.size(0)
        device = x.device

        # --- Tracking Initialization ---
        pre_activations_tracking = []
        post_activations_tracking = []
        synch_out_tracking = []
        synch_action_tracking = []
        attention_tracking = []

        # --- Featurise Input Data ---
        kv = self.compute_features(x)

        # --- Initialise Recurrent State ---
        state_trace = self.start_trace.unsqueeze(0).expand(B, -1, -1) # Shape: (B, H, T)
        activated_state = self.start_activated_state.unsqueeze(0).expand(B, -1) # Shape: (B, H)

        # --- Storage for outputs per iteration
        predictions = torch.empty(B, self.out_dims, self.iterations, device=device, dtype=x.dtype)
        certainties = torch.empty(B, 2, self.iterations, device=device, dtype=x.dtype)

        decay_alpha_action, decay_beta_action = None, None
        r_action, r_out = torch.exp(-self.decay_params_action).unsqueeze(0).repeat(B, 1), torch.exp(-self.decay_params_out).unsqueeze(0).repeat(B, 1)

        _, decay_alpha_out, decay_beta_out = self.compute_synchronisation(activated_state, None, None, r_out, synch_type='out')

        # --- Recurrent Loop  ---
        for stepi in range(self.iterations):

            # --- Calculate Synchronisation for Input Data Interaction ---
            synchronisation_action, decay_alpha_action, decay_beta_action = self.compute_synchronisation(activated_state, decay_alpha_action, decay_beta_action, r_action, synch_type='action')

            # --- Interact with Data via Attention ---
            q = self.q_proj(synchronisation_action).unsqueeze(1)
            attn_out, attn_weights = self.attention(q, kv, kv, average_attn_weights=False, need_weights=True)
            attn_out = attn_out.squeeze(1)
            pre_synapse_input = torch.concatenate((attn_out, activated_state), dim=-1)

            # --- Apply Synapses ---
            state = self.synapses(pre_synapse_input)
            state_trace = torch.cat((state_trace[:, :, 1:], state.unsqueeze(-1)), dim=-1)

            # --- Activate ---
            activated_state = self.trace_processor(state_trace)

            # --- Calculate Synchronisation for Output Predictions ---
            synchronisation_out, decay_alpha_out, decay_beta_out = self.compute_synchronisation(activated_state, decay_alpha_out, decay_beta_out, r_out, synch_type='out')

            # --- Get Predictions and Certainties ---
            current_prediction = self.output_projector(synchronisation_out)
            current_certainty = self.compute_certainty(current_prediction)

            predictions[..., stepi] = current_prediction
            certainties[..., stepi] = current_certainty

            # --- Tracking ---
            if track:
                pre_activations_tracking.append(state_trace[:,:,-1].detach().cpu().numpy())
                post_activations_tracking.append(activated_state.detach().cpu().numpy())
                attention_tracking.append(attn_weights.detach().cpu().numpy())
                synch_out_tracking.append(synchronisation_out.detach().cpu().numpy())
                synch_action_tracking.append(synchronisation_action.detach().cpu().numpy())

        # --- Return Values ---
        if track:
            return predictions, certainties, (np.array(synch_out_tracking), np.array(synch_action_tracking)), np.array(pre_activations_tracking), np.array(post_activations_tracking), np.array(attention_tracking)
        return predictions, certainties, synchronisation_out


def get_loss(predictions, certainties, targets, use_most_certain=True):
    """use_most_certain will select either the most certain point or the final point."""
    losses = nn.CrossEntropyLoss(reduction='none')(predictions,
                                                   torch.repeat_interleave(targets.unsqueeze(-1), predictions.size(-1), -1))
    loss_index_1 = losses.argmin(dim=1)
    loss_index_2 = certainties[:,1].argmax(-1)
    if not use_most_certain:
        loss_index_2[:] = -1
    batch_indexer = torch.arange(predictions.size(0), device=predictions.device)
    loss_minimum_ce = losses[batch_indexer, loss_index_1].mean()
    loss_selected = losses[batch_indexer, loss_index_2].mean()
    loss = (loss_minimum_ce + loss_selected)/2
    return loss, loss_index_2


def calculate_accuracy(predictions, targets, where_most_certain):
    """Calculate the accuracy based on the prediction at the most certain internal tick."""
    B = predictions.size(0)
    dev = predictions.device
    predictions_at_most_certain_internal_tick = predictions.argmax(1)[torch.arange(B, device=dev), where_most_certain].detach().cpu().numpy()
    accuracy = (targets.detach().cpu().numpy() == predictions_at_most_certain_internal_tick).mean()
    return accuracy


print('official model source loaded verbatim.')
print('official model source loaded verbatim.')

### Data and constants

MNIST and CIFAR-10 loaders, seeding, and the shared thought budget.

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

D, T = 128, 20
N_SUB = 16                            # n_synch_out -- the subspace that actually feeds the readout
N_CHECK = 20
ALIGN_TICKS = list(range(1, T + 1))      # [8.4]: every tick (reports the peak and the crossing)
DATASETS = ['mnist', 'cifar']

CTM_KWARGS = dict(iterations=T, d_model=D, d_input=D, memory_length=15, heads=2,
                  n_synch_out=N_SUB, n_synch_action=N_SUB, memory_hidden_dims=16, out_dims=10, dropout=0.0)

loaders = {}
tfm_m = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
tr_m = datasets.MNIST('./data', train=True, download=True, transform=tfm_m)
te_m = datasets.MNIST('./data', train=False, download=True, transform=tfm_m)
loaders['mnist'] = dict(
    train=torch.utils.data.DataLoader(tr_m, batch_size=128, shuffle=True, num_workers=2, drop_last=True),
    test=torch.utils.data.DataLoader(te_m, batch_size=128, shuffle=False, num_workers=2))

if not os.path.exists('./CIFAR-10-images'):
    print('Cloning CIFAR-10-images repository...')
    import subprocess
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/YoongiKim/CIFAR-10-images.git'], check=True)
_norm_c = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
tfm_c = transforms.Compose([transforms.ToTensor(), _norm_c])          # eval transform (never augmented)
tfm_c_train = transforms.Compose(                                      # train transform
    ([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()] if CIFAR_AUGMENT else [])
    + [transforms.ToTensor(), _norm_c])
tr_c = datasets.ImageFolder('./CIFAR-10-images/train', transform=tfm_c_train)
te_c_raw = datasets.ImageFolder('./CIFAR-10-images/test', transform=tfm_c)
perm = torch.randperm(len(te_c_raw), generator=torch.Generator().manual_seed(0)).tolist()
te_c = torch.utils.data.Subset(te_c_raw, perm)
loaders['cifar'] = dict(
    train=torch.utils.data.DataLoader(tr_c, batch_size=128, shuffle=True, num_workers=2, drop_last=True),
    test=torch.utils.data.DataLoader(te_c, batch_size=128, shuffle=False, num_workers=2))
print('data ready')

### A stronger CIFAR-10 backbone

A ResNet-9 replaces the original small backbone, lifting CIFAR accuracy to $0.85$ from $0.64$ and making the model a competent classifier rather than a toy (§8).

In [ ]:
class ResidualBlock(nn.Module):
    """Standard pre-activation-free residual block: two 3x3 convs, identity skip."""
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return F.relu(x + h)

def _conv_bn(i, o, pool=False):
    layers = [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return layers

def make_resnet9_backbone(d_input, in_ch=3):
    """32x32 -> 8x8 with d_input channels: same token count and width the official backbone produces,
    so nothing downstream of the backbone changes shape."""
    return nn.Sequential(
        *_conv_bn(in_ch, 64),               # 32x32
        *_conv_bn(64, 128, pool=True),      # 16x16
        ResidualBlock(128),
        *_conv_bn(128, 256, pool=True),     #  8x8
        ResidualBlock(256),
        *_conv_bn(256, d_input),            #  8x8, back to d_input channels
        ResidualBlock(d_input),
    )

def maybe_upgrade_backbone(model, name):
    """Swap in the stronger stack for CIFAR-10 only, and only when selected."""
    if name == 'cifar' and CIFAR_BACKBONE == 'resnet9':
        model.backbone = make_resnet9_backbone(model.d_input, in_ch=3).to(device)
    return model

# CIFAR-only schedule override -- the official schedule underfits the larger backbone
CIFAR_LR_BY_BACKBONE    = {'official': 1e-4, 'resnet9': 3e-4}

def _n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"CIFAR backbone: {CIFAR_BACKBONE}")
print(f"  steps {CIFAR_STEPS}, lr {CIFAR_LR_BY_BACKBONE[CIFAR_BACKBONE]}, "
      f"augmentation {CIFAR_AUGMENT}")
if CIFAR_BACKBONE == 'resnet9':
    _bb = make_resnet9_backbone(D)
    print(f"  backbone parameters: {_n_params(_bb):,} (official backbone is ~0.15M)")
    print("  D, T, n_synch unchanged -> the numbers of §8 stay directly comparable")
    del _bb

### Decision tick

`get_decision_tick` — the tick of highest certainty. Every statistic below is computed over $[1,t_{\mathrm{dec}}]$.

In [ ]:
def get_decision_tick(predictions, certainties):
    """Per-sample tick index used as 'the decision', 0-indexed. predictions: (B,C,T), certainties: (B,2,T)."""
    if DECISION_MODE == 'most_certain_tick':
        return certainties[:, 1, :].argmax(-1)
    elif DECISION_MODE == 'last_tick':
        B, Tt = predictions.shape[0], predictions.shape[-1]
        return torch.full((B,), Tt - 1, dtype=torch.long, device=predictions.device)
    raise ValueError(DECISION_MODE)

def decision_prediction(predictions, certainties):
    """(predicted_class, decision_tick) per sample."""
    t_dec = get_decision_tick(predictions, certainties)
    B = predictions.shape[0]
    pred = predictions.argmax(1)[torch.arange(B, device=predictions.device), t_dec]
    return pred, t_dec

# Part 2 · Train or load the two models

Three seeds each for MNIST and CIFAR. Checkpoints are reused when present, so this cell is cheap on re-runs. **The accuracies printed here are the `acc.` column of Table 2.**

In [ ]:
STEPS = {'mnist': 1500, 'cifar': CIFAR_STEPS}
LR    = {'mnist': 1e-4,  'cifar': CIFAR_LR_BY_BACKBONE[CIFAR_BACKBONE]}
# ---- where checkpoints live ---------------------------------------------------------------------
import shutil, hashlib

BASE_DIR = LOCAL_DIR
if USE_DRIVE:
    try:
        from google.colab import drive as _gdrive
        _gdrive.mount('/content/drive')
        _root = '/content/drive/MyDrive'
        if not os.path.isdir(os.path.dirname(DRIVE_DIR)):
            raise RuntimeError(f'{os.path.dirname(DRIVE_DIR)} is not a directory')
        if not os.path.isdir(DRIVE_DIR):
            print(f'  [drive] {DRIVE_DIR} does not exist yet. Folders currently in MyDrive/:')
            for _d in sorted(d for d in os.listdir(_root) if os.path.isdir(os.path.join(_root, d)))[:40]:
                print(f'            {_d}')
            print('  [drive] creating it. If you meant an existing folder above, set DRIVE_DIR to it.')
        BASE_DIR = DRIVE_DIR
    except Exception as _e:
        print(f'  [drive] unavailable ({type(_e).__name__}: {_e}); using {LOCAL_DIR}')

CKPT_DIR = os.path.join(BASE_DIR, 'ckpts')
FIG_DIR  = os.path.join(BASE_DIR, 'figures')
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print(f'  [ckpt] checkpoints -> {CKPT_DIR}')
print(f'  [figs] figures     -> {FIG_DIR}')


def fig_path(fname):
    """Every figure the manuscript uses is written here, so a runtime reset never loses them again."""
    return os.path.join(FIG_DIR, fname)


def _ckpt_cfg(name, kind, seed):
    """Everything that changes what the trained weights ARE. Stored in the file and checked on load,
    so a changed backbone or budget can never be silently served from a stale checkpoint."""
    return dict(dataset=name, kind=kind, seed=seed,
                backbone=(CIFAR_BACKBONE if name == 'cifar' else 'official'),
                steps=STEPS[name], lr=LR[name],
                augment=(CIFAR_AUGMENT if name == 'cifar' else False),
                ctm=dict(sorted((k, str(v)) for k, v in CTM_KWARGS.items())))


def _save_ckpt(model, path, cfg):
    """Write locally first, then copy. Writing a large tensor straight onto the Drive FUSE mount can
    flush partially and leave a file that loads as garbage."""
    tmp = os.path.join('/tmp', os.path.basename(path))
    torch.save({'state_dict': model.state_dict(), 'cfg': cfg}, tmp)
    shutil.copy(tmp, path)
    os.remove(tmp)


def _load_ckpt(model, path, cfg):
    """Returns the model, or None if the checkpoint does not match the config we asked for."""
    blob = torch.load(path, map_location=device, weights_only=False)
    if not (isinstance(blob, dict) and 'state_dict' in blob):
        blob = {'state_dict': blob, 'cfg': None}          # accept pre-versioning checkpoints
    if blob['cfg'] is not None and blob['cfg'] != cfg:
        differing = [k for k in cfg if blob['cfg'].get(k) != cfg[k]]
        print(f'  [ckpt] {os.path.basename(path)} config mismatch on {differing}; retraining')
        return None
    model.load_state_dict(blob['state_dict']); model.eval()
    return model

def build_model(name='mnist'):
    m = ContinuousThoughtMachine(**CTM_KWARGS).to(device)
    m = maybe_upgrade_backbone(m, name)      # CIFAR-10 only, and only if CIFAR_BACKBONE=='resnet9'
    def clamp_decay_params(module, _input):
        with torch.no_grad():
            module.decay_params_action.data.clamp_(0, 15)
            module.decay_params_out.data.clamp_(0, 15)
    m.register_forward_pre_hook(clamp_decay_params)
    return m

def materialize(m, name):
    x0, _ = next(iter(loaders[name]['train']))
    with torch.no_grad():
        m(x0.to(device), track=False)
    return m

def train(model, loader, steps, lr=1e-4, tag='', shuffle_labels=False):
    opt = torch.optim.AdamW(list(model.parameters()), lr=lr, eps=1e-8)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    model.train(); it = iter(loader); t0 = time.time()
    for stepi in range(steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        if shuffle_labels:
            y = y[torch.randperm(len(y), device=device)]
        predictions, certainties, _ = model(x, track=False)
        loss, wmc = get_loss(predictions, certainties, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if (stepi + 1) % max(steps // 4, 1) == 0:
            print(f'    [{tag}] step {stepi+1}/{steps}  acc '
                  f'{calculate_accuracy(predictions, y, wmc):.3f}  ({time.time()-t0:.0f}s)')

def get_or_train(name, kind, seed):
    # checkpoints are tagged by backbone so the two CIFAR variants never collide
    # the canonical config keeps the original untagged filename, so checkpoints from earlier runs are
    # reused rather than retrained; any other backbone/budget gets its own tag
    canonical = (CIFAR_BACKBONE == 'official' and CIFAR_STEPS == 6000)
    tag_bb = '' if (name != 'cifar' or canonical) else f'_{CIFAR_BACKBONE}_{CIFAR_STEPS}'
    if kind == 'frozen':
        set_seed(400 + seed); m = materialize(build_model(name), name); m.eval(); return m
    path = os.path.join(CKPT_DIR, f'{name}{tag_bb}_{kind}_s{seed}.pt')
    cfg  = _ckpt_cfg(name, kind, seed)
    if (not FORCE_RETRAIN) and os.path.exists(path):
        m = _load_ckpt(materialize(build_model(name), name), path, cfg)
        if m is not None:
            print(f'  [{name}-{kind}-s{seed}] loaded from checkpoint'); return m
    set_seed((200 if kind == 'real' else 300) + seed)
    print(f'== {name} seed {seed}: {kind} ==')
    m = build_model(name)
    train(m, loaders[name]['train'], STEPS[name], lr=LR[name], tag=f'{name}{tag_bb}-{kind}-s{seed}',
          shuffle_labels=(kind == 'shuffled'))
    m.eval(); _save_ckpt(m, path, cfg); return m

MODELS = {n: {k: [] for k in ('real', 'shuffled', 'frozen')} for n in DATASETS}
_t0 = time.time()
for name in DATASETS:
    for seed in range(N_SEEDS):
        for kind in ('real', 'shuffled', 'frozen'):
            MODELS[name][kind].append(get_or_train(name, kind, seed))
print(f'\nall models ready ({time.time()-_t0:.0f}s)')

RESULTS = {}
def mean_sd(vals):
    a = np.asarray(vals, dtype=float); return float(np.nanmean(a)), float(np.nanstd(a))

# --- held-out test accuracy of each real model, so 'did the backbone help?' is answered directly ---
@torch.no_grad()
def test_accuracy(model, name, n_batches=16):
    correct = total = 0
    it = iter(loaders[name]['test'])
    for _ in range(n_batches):
        try:
            x, y = next(it)
        except StopIteration:
            break
        preds, certs, _ = model(x.to(device), track=False)
        pred, _ = decision_prediction(preds.cpu(), certs.cpu())
        correct += (pred == y).sum().item(); total += len(y)
    return correct / max(total, 1)

print('\nheld-out test accuracy (real models, decision tick):')
for name in DATASETS:
    accs = [test_accuracy(MODELS[name]['real'][s], name) for s in range(N_SEEDS)]
    bb = CIFAR_BACKBONE if name == 'cifar' else 'official'
    RESULTS_ACC = globals().setdefault('RESULTS_ACC', {})
    RESULTS_ACC[name] = (float(np.mean(accs)), float(np.std(accs)))
    print(f'  {name.upper():6s} [{bb:8s}]  {np.mean(accs):.4f} +- {np.std(accs):.4f}')

### The tick map and the readout entropy

`run_to_tick` (the context $\mathcal{A}^t$), `one_tick_step` (the frozen map $f(\cdot;\mathcal{A}^t)$), `entropy_of_z` (normalized entropy of the readout from a state), and `zero_crossing` (the tick at which an alignment curve reverses for good). Used by [8.1]–[8.4].

In [ ]:
@torch.no_grad()
# WHAT IS IN THE CONTEXT A^t. run_to_tick returns the `frozen` dict, which is exactly the context
# the one-step map consumes besides the state itself. Three things vary or matter:
#
#   kv                                the token bank, computed ONCE from the image; constant in t
#   decay_alpha_action, _beta_action  the action-side accumulator -> the query (Eq. 4)
#   state_trace                       the last M pre-activations per neuron -> the neuron models
#   r_action, r_out                   learned decay rates; constant in t
#   decay_alpha_out, _beta_out        the OUT-side accumulator -- present in the dict but NOT read
#                                     by one_tick_step; it is used only by entropy_of_z
#
# So f varies with t through exactly two objects, both of them memories of the trajectory: a decayed
# accumulation over all past ticks, and a hard window of length M. The statistic the prediction is
# read from plays no part in producing the next state.
def run_to_tick(model, x, t_stop):
    B = x.size(0)
    kv = model.compute_features(x)
    state_trace = model.start_trace.unsqueeze(0).expand(B, -1, -1).clone()
    activated_state = model.start_activated_state.unsqueeze(0).expand(B, -1).clone()
    decay_alpha_action, decay_beta_action = None, None
    r_action = torch.exp(-model.decay_params_action).unsqueeze(0).repeat(B, 1)
    r_out = torch.exp(-model.decay_params_out).unsqueeze(0).repeat(B, 1)
    _, decay_alpha_out, decay_beta_out = model.compute_synchronisation(activated_state, None, None, r_out, synch_type='out')
    for stepi in range(t_stop):
        sync_a, decay_alpha_action, decay_beta_action = model.compute_synchronisation(
            activated_state, decay_alpha_action, decay_beta_action, r_action, synch_type='action')
        q = model.q_proj(sync_a).unsqueeze(1)
        attn_out, _ = model.attention(q, kv, kv, average_attn_weights=False, need_weights=True)
        attn_out = attn_out.squeeze(1)
        state = model.synapses(torch.cat((attn_out, activated_state), dim=-1))
        state_trace = torch.cat((state_trace[:, :, 1:], state.unsqueeze(-1)), dim=-1)
        activated_state = model.trace_processor(state_trace)
        _, decay_alpha_out, decay_beta_out = model.compute_synchronisation(activated_state, decay_alpha_out, decay_beta_out, r_out, synch_type='out')
    # The out-side accumulators (decay_alpha_out, decay_beta_out, r_out) are returned as well as the
    # action-side ones: entropy_of_z needs the out-side EMA as it stood BEFORE the tick in order to
    # define H_t as a pure function of z.
    return dict(activated_state=activated_state, state_trace=state_trace, kv=kv,
               decay_alpha_action=decay_alpha_action, decay_beta_action=decay_beta_action, r_action=r_action,
               decay_alpha_out=decay_alpha_out, decay_beta_out=decay_beta_out, r_out=r_out)

def one_tick_step(model, z, frozen):
    sync_a, _, _ = model.compute_synchronisation(z, frozen['decay_alpha_action'], frozen['decay_beta_action'], frozen['r_action'], synch_type='action')
    q = model.q_proj(sync_a).unsqueeze(1)
    attn_out, _ = model.attention(q, frozen['kv'], frozen['kv'], average_attn_weights=False, need_weights=True)
    attn_out = attn_out.squeeze(1)
    new_state = model.synapses(torch.cat((attn_out, z), dim=-1))
    new_state_trace = torch.cat((frozen['state_trace'][:, :, 1:], new_state.unsqueeze(-1)), dim=-1)
    return model.trace_processor(new_state_trace)

def entropy_of_z(model, z, frozen_prev):
    """H_t(z): normalized entropy of the readout built from z and the OUT-side accumulator as it stood
    BEFORE tick t. A pure function of z -- no future information leaks in."""
    sync_o, _, _ = model.compute_synchronisation(z, frozen_prev['decay_alpha_out'],
                                                 frozen_prev['decay_beta_out'], frozen_prev['r_out'],
                                                 synch_type='out')
    return model.compute_certainty(model.output_projector(sync_o))[..., 0]

def zero_crossing(curve, ticks):
    """Tick at which the alignment reverses for good: the first tick AFTER the curve's peak at
    which it is negative and the remainder of the curve is negative on average.

    Two failure modes this avoids. Anchoring to the peak is what stops a plain first-crossing rule
    from firing on a transient wobble around zero in the opening ticks, before the descending phase
    has begun -- whether it does so depends only on the sign of tick 1. And testing the mean of the
    tail rather than every value in it is what keeps the rule from returning None at higher
    capacity, where the alignment reverses and then drifts back to about zero by the final tick,
    leaving a genuine reversal but no all-negative tail."""
    curve = np.asarray(curve)
    k0 = int(np.argmax(curve))
    for k in range(k0 + 1, len(curve)):
        if curve[k] < 0 and curve[k:].mean() < 0:
            return ticks[k]
    return None

# Part 3 · §8 · Smaller models

### [8.1] Table 2 · $\rho(J)$ and $\sigma_{\max}(J)$ at the fixed point

Power iteration at $z^*$, 16 test images × the ticks of `_cticks(T)` per seed, 3 seeds, measured as for ImageNet. Fills the $\rho(J)$ and $\sigma_{\max}(J)$ columns of Table 2: $\rho<1$ on both (the attracting curve is universal), while $\sigma_{\max}$ stays below one on MNIST and in the median on CIFAR (Euclidean contractions).

In [ ]:
# [8.1] Table 2 · rho(J) and sigma_max(J) at z* by power iteration -> SPECTRA
from torch.autograd.functional import jvp as _AJVP, vjp as _AVJP
import numpy as np, math, torch

RHO_ITERS, SIG_ITERS, FP_ITERS, N_CONTRACT_S = 80, 18, 200, 16
def _cticks(T_): return list(range(2, T_, max(1, T_ // 12)))

def _spectral_radius(g, z, iters=RHO_ITERS, seed=0):
    z = z.detach(); torch.manual_seed(seed)
    w = torch.randn_like(z); w = w / (w.norm() + 1e-12); logs = []
    with torch.enable_grad():
        for _ in range(iters):
            _, Jw = _AJVP(g, (z,), (w,)); f = float(Jw.norm())
            logs.append(math.log(f + 1e-30)); w = (Jw / (f + 1e-12)).detach()
    return float(math.exp(np.mean(logs[len(logs) // 2:])))

def _sigma_max(g, z, iters=SIG_ITERS, seed=0):
    z = z.detach(); torch.manual_seed(seed)
    v = torch.randn_like(z); v = v / (v.norm() + 1e-12); s = 0.0
    with torch.enable_grad():
        for _ in range(iters):
            _, Jv = _AJVP(g, (z,), (v,)); _, JtJv = _AVJP(g, (z,), Jv)
            JtJv = JtJv[0] if isinstance(JtJv, tuple) else JtJv
            n = JtJv.norm(); s = float(n.sqrt().item()); v = (JtJv / (n + 1e-12)).detach()
    return s

@torch.no_grad()
def _fixed_point_s(model, frozen, z0, iters=FP_ITERS, tol=1e-8):
    z = z0.clone()
    for _ in range(iters):
        zn = one_tick_step(model, z, frozen)
        if float((zn - z).norm()) < tol: z = zn; break
        z = zn
    return z

def spectra_at_zstar(model, xb, ticks):
    rho, sig = [], []
    for i in range(xb.shape[0]):
        x1 = xb[i:i+1]
        for t in ticks:
            with torch.no_grad(): fr = run_to_tick(model, x1, t)
            z = fr['activated_state'].detach()
            with torch.no_grad(): zstar = _fixed_point_s(model, fr, z)
            g = lambda zz: one_tick_step(model, zz, fr)
            rho.append(_spectral_radius(g, zstar)); sig.append(_sigma_max(g, zstar))
    return np.array(rho), np.array(sig)

print(f'{"dataset":8s} {"backbone":9s}  rho@z* med/max     sigma_max@z* med/max   sigma/rho')
SPECTRA = {}
for name in DATASETS:
    R, S = [], []
    for s in range(N_SEEDS):
        model = MODELS[name]['real'][s]
        xb, _ = next(iter(loaders[name]['test'])); xb = xb[:N_CONTRACT_S].to(device)
        r, sg = spectra_at_zstar(model, xb, _cticks(model.iterations)); R.append(r); S.append(sg)
    R, S = np.concatenate(R), np.concatenate(S); SPECTRA[name] = (R, S)
    bb = (CIFAR_BACKBONE if 'cifar' in name else 'official')
    print(f'{name:8s} {bb:9s}  {np.median(R):.3f}/{R.max():.3f}      '
          f'{np.median(S):.3f}/{S.max():.3f}        {np.median(S/np.clip(R,1e-6,None)):.1f}x')
for name in DATASETS:
    R, S_ = SPECTRA[name]
    print(f'{name:8s} sigma_max < 1 at {(S_ < 1).mean():.0%} of fixed points')

### [8.2] Non-normality at initialization and after training

Four versions of each model, read at $z^*$ with the exact $128\times128$ Jacobian: **init** (untrained), **random core** (trained backbone, recurrence at init), **shuffled labels** (same recipe, no learnable task) and **trained**. Prints $\rho$, $\sigma_{\max}$, $\sigma_{\max}/\rho$ and the normal share $\sum|\lambda|^2/\lVert J\rVert_F^2$ (1 = normal, ≈ 0.5 = Ginibre, 0 = pure feed-forward cascade), against a same-size Ginibre matrix. Numbers only; a few minutes on a T4.

In [ ]:
# [8.2] §8 · do the small CTMs start non-normal, and what does training do to it? ──────────────────
# Four versions of each model, all read at the fixed point z*(A^t) with the same frozen-context map:
#   init             fully untrained (the existing `frozen` models)
#   random core      trained backbone + kv_proj of the real model, recurrence at init (matched input)
#   shuffled labels  trained with the same recipe on shuffled labels (training without a learnable task)
#   trained          the real models
# D = 128, so everything is exact: the full Jacobian J at z*, its eigenvalues and singular values.
#   rho        spectral radius               (attraction: < 1)
#   sigma_max  operator norm                 (transient amplification possible iff > 1)
#   sigma/rho  worst-direction non-normality (1 = normal, ~2 = Ginibre)
#   normal share  sum|lambda|^2 / ||J||_F^2 = 1 - departure^2   (1 = normal, 0.5 = Ginibre, 0 = pure cascade)
#   newest slot   share of each neuron model's window sensitivity on the newest entry (uniform = 1/M)
# Cost: a few minutes on a T4 (24 models, batched exact Jacobians).
from tqdm.auto import tqdm

N_NN = 16                                  # test images per model
NN_TICKS = _cticks(T)                      # same ticks as [8.1]
KINDS = ('init', 'random core', 'shuffled labels', 'trained')
INPUT_PATH = ('backbone', 'kv_proj')
# ImageNet reference (ImageNet notebook [3.4] and Table 3)
IMAGENET_REF = {'random core': dict(sr=7.1, ns=1 - 0.932**2), 'trained': dict(sr=24.4, ns=1 - 0.991**2),
                'init': dict(sr=6.9, ns=np.nan)}

def _random_core(name, seed):
    set_seed(500 + seed)
    m = materialize(build_model(name), name)
    src = {k: v for k, v in MODELS[name]['real'][seed].state_dict().items() if k.split('.')[0] in INPUT_PATH}
    m.load_state_dict(src, strict=False); m.eval()
    return m

def _model(name, kind, seed):
    return {'init': lambda: MODELS[name]['frozen'][seed], 'random core': lambda: _random_core(name, seed),
            'shuffled labels': lambda: MODELS[name]['shuffled'][seed],
            'trained': lambda: MODELS[name]['real'][seed]}[kind]()

def _exact_J_batch(m, fr, zstar):
    """Per-sample Jacobians at z*. The map is sample-independent once kv is computed, so the
    Jacobian of the batched map is block-diagonal; we keep the diagonal blocks."""
    B = zstar.shape[0]
    F_ = lambda Z: one_tick_step(m, Z, fr)
    with torch.enable_grad():
        Jf = torch.autograd.functional.jacobian(F_, zstar, vectorize=True)      # (B, D, B, D)
    idx = torch.arange(B)
    return Jf[idx, :, idx, :].double()                                         # (B, D, D)

def _window_share(m, A):
    """|dz_i/dA_{i,k}| per neuron and window slot (NLMs act per neuron), normalised over slots."""
    cols = []
    for k in range(A.shape[-1]):
        e = torch.zeros_like(A); e[..., k] = 1.0
        with torch.enable_grad(): cols.append(_AJVP(m.trace_processor, (A,), (e,))[1].abs())
    S = torch.stack(cols, -1)
    return (S / S.sum(-1, keepdim=True).clamp_min(1e-12)).flatten(0, 1).cpu().numpy()   # (B·D, M)

def nonnormality(m, xb):
    rho, sig, ns, conv, prof = [], [], [], [], []
    for t in NN_TICKS:
        with torch.no_grad():
            fr = run_to_tick(m, xb, t)
            zs = _fixed_point_s(m, fr, fr['activated_state'])
            conv.append(((one_tick_step(m, zs, fr) - zs).norm(dim=-1) / zs.norm(dim=-1)).cpu().numpy())
        J = _exact_J_batch(m, fr, zs.detach())
        lam = torch.linalg.eigvals(J).abs(); sv = torch.linalg.svdvals(J)
        rho.append(lam.max(-1).values.cpu().numpy()); sig.append(sv[:, 0].cpu().numpy())
        ns.append(((lam**2).sum(-1) / (sv**2).sum(-1)).cpu().numpy())         # ||J||_F^2 = sum sigma^2
        prof.append(_window_share(m, fr['state_trace'].detach()))
    cat = np.concatenate
    return dict(rho=cat(rho), sig=cat(sig), ns=cat(ns), conv=cat(conv), profile=cat(prof).mean(0),
                newest=float(np.median(cat(prof)[:, -1])))

# Ginibre reference at the same size (exact)
_g = [np.random.default_rng(k).standard_normal((D, D)) for k in range(50)]
GIN = dict(sr=np.median([np.linalg.svd(G, compute_uv=False)[0] / np.abs(np.linalg.eigvals(G)).max() for G in _g]),
           ns=np.median([(np.abs(np.linalg.eigvals(G))**2).sum() / (G**2).sum() for G in _g]))

NN = {name: {k: [] for k in KINDS} for name in DATASETS}
jobs = [(n, k, s) for n in DATASETS for k in KINDS for s in range(N_SEEDS)]
t0 = time.time()
for name, kind, s in tqdm(jobs, desc='models', unit='model'):
    m = _model(name, kind, s)
    xb, _ = next(iter(loaders[name]['test'])); xb = xb[:N_NN].to(device)
    r = nonnormality(m, xb); r['acc'] = test_accuracy(m, name, n_batches=4)
    NN[name][kind].append(r)
    if kind == 'random core': del m; torch.cuda.empty_cache()
print(f'done in {time.time() - t0:.0f}s · {N_NN} images × {len(NN_TICKS)} ticks per model · {N_SEEDS} seeds')

# ---- table ----------------------------------------------------------------------------------------
M = MODELS[DATASETS[0]]['real'][0].memory_length
pool = lambda name, kind, key: np.concatenate([np.atleast_1d(r[key]) for r in NN[name][kind]])
SUMM = {}
print(f"\nExact spectrum of J at z*  (medians; Ginibre {D}×{D}: σ/ρ {GIN['sr']:.2f}, normal share {GIN['ns']:.2f})")
hdr = (f"{'':16s} {'top-1':>6s} {'ρ':>7s} {'σmax':>7s} {'σ<1':>6s} {'σ/ρ':>6s} {'normal share':>13s}"
       f" {'newest slot':>12s} {'FP conv.':>9s}")
for name in DATASETS:
    print(f"\n{name.upper()}" + (f'  [backbone: {CIFAR_BACKBONE}]' if name == 'cifar' else ''))
    print(hdr)
    SUMM[name] = {}
    for kind in KINDS:
        rho, sig, ns = pool(name, kind, 'rho'), pool(name, kind, 'sig'), pool(name, kind, 'ns')
        d = dict(rho=np.median(rho), sig=np.median(sig), sr=np.median(sig / rho), ns=np.median(ns),
                 ns_iqr=np.percentile(ns, [25, 75]), sr_iqr=np.percentile(sig / rho, [25, 75]),
                 fs=(sig < 1).mean(), newest=np.mean([r['newest'] for r in NN[name][kind]]),
                 profile=np.mean([r['profile'] for r in NN[name][kind]], 0),
                 acc=np.mean([r['acc'] for r in NN[name][kind]]),
                 conv=(pool(name, kind, 'conv') < 1e-4).mean())
        SUMM[name][kind] = d
        print(f"{kind:16s} {d['acc']:6.3f} {d['rho']:7.3f} {d['sig']:7.3f} {d['fs']:6.0%} {d['sr']:6.1f}"
              f" {d['ns']:13.3f} {d['newest']:12.3f} {d['conv']:9.0%}")
print(f"\n(normal share: 1 = normal, {GIN['ns']:.2f} = Ginibre, 0 = pure feed-forward cascade;"
      f" newest slot: uniform = 1/M = {1/M:.3f})")

# ---- verdict --------------------------------------------------------------------------------------
def _move(a, b, lower_is_more=False):
    f = (a / b) if lower_is_more else (b / a)
    return f, ('MORE non-normal' if f > 1.5 else 'MORE normal' if f < 1 / 1.5 else 'unchanged')

print('\nVERDICT')
for name in DATASETS:
    S = SUMM[name]; i, rc, sh, tr = S['init'], S['random core'], S['shuffled labels'], S['trained']
    start = ('non-normal beyond Ginibre' if i['sr'] > 1.5 * GIN['sr'] or i['ns'] < 0.7 * GIN['ns']
             else 'Ginibre-like')
    f_ns, w_ns = _move(rc['ns'], tr['ns'], lower_is_more=True)
    print(f"  {name.upper():6s} starts {start} (init σ/ρ {i['sr']:.1f}, normal share {i['ns']:.2f}).")
    print(f"         training vs matched random core: normal share {rc['ns']:.3f} → {tr['ns']:.3f} "
          f"({f_ns:.1f}× → {w_ns}); σ/ρ {rc['sr']:.1f} → {tr['sr']:.1f}; σ<1 {rc['fs']:.0%} → {tr['fs']:.0%}.")
    if sh['rho'] < 0.01:
        print(f"         shuffled labels: DEGENERATE map (ρ {sh['rho']:.3f}, σmax {sh['sig']:.3f}: J ≈ 0) → "
              f"its non-normality ratios are not comparable.")
    else:
        share = np.log(rc['ns'] / sh['ns']) / np.log(rc['ns'] / tr['ns']) if w_ns != 'unchanged' else np.nan
        print(f"         shuffled labels: normal share → {sh['ns']:.3f}, σ/ρ {sh['sr']:.1f} → reproduces "
              f"{share:.0%} of the trained change (log scale)" + (' → mostly a TRAINING effect, not task-specific.'
              if share > 0.5 else ' → mostly needs the TASK.' if share < 0.25 else ' → partly task, partly training.'))
ir, it = IMAGENET_REF['random core'], IMAGENET_REF['trained']
print(f"  IMAGENET (reference): normal share {ir['ns']:.3f} → {it['ns']:.3f} ({ir['ns']/it['ns']:.1f}× → MORE non-normal); "
      f"σ/ρ {ir['sr']:.1f} → {it['sr']:.1f}.")

### [8.3] Table 2 · Attention entropy and the query ablations

Top-1 at $t_{\mathrm{dec}}$ (6 test batches per seed, 3 seeds) when the query is frozen at its first-tick value, swapped with another image's query, when attention is made uniform, or removed; and the entropy of the head-averaged attention weights as a fraction of $\ln L$, with $L$ read from the attention tensor. Fills the `attn.`, `frozen` and `shuffled` columns of Table 2.

In [ ]:
# [8.3] Table 2 · attention entropy and the query/attention ablations -> ABLATE
# The loop below reproduces the model's own forward pass exactly (same accumulator recursions, same
# trace update), with one hook: the query fed to attention can be overridden per tick.

@torch.no_grad()
def rollout(model, x, mode='intact', gen=None):
    """mode: 'intact' | 'frozen' (q^1 throughout) | 'shuffled' (another sample's query)."""
    B = x.size(0)
    kv = model.compute_features(x)
    state_trace = model.start_trace.unsqueeze(0).expand(B, -1, -1)
    activated_state = model.start_activated_state.unsqueeze(0).expand(B, -1)
    predictions = torch.empty(B, model.out_dims, model.iterations, device=x.device, dtype=x.dtype)
    certainties = torch.empty(B, 2, model.iterations, device=x.device, dtype=x.dtype)

    da_a, db_a = None, None
    r_a = torch.exp(-model.decay_params_action).unsqueeze(0).repeat(B, 1)
    r_o = torch.exp(-model.decay_params_out).unsqueeze(0).repeat(B, 1)
    _, da_o, db_o = model.compute_synchronisation(activated_state, None, None, r_o, synch_type='out')

    q_first = None
    diag = dict(w_max=[], w_ent=[], a_norm=[], z_norm=[])
    for stepi in range(model.iterations):
        sync_a, da_a, db_a = model.compute_synchronisation(activated_state, da_a, db_a, r_a,
                                                          synch_type='action')
        q = model.q_proj(sync_a)
        if stepi == 0:
            q_first = q.clone()
        if mode == 'frozen':
            q = q_first                                   # never adapts to the image
        elif mode == 'shuffled':
            perm = torch.randperm(B, generator=gen, device=q.device)
            q = q[perm]                                   # right distribution, wrong image

        attn_out, attn_w = model.attention(q.unsqueeze(1), kv, kv, average_attn_weights=True,
                                           need_weights=True)
        attn_out = attn_out.squeeze(1)
        if mode == 'uniform':
            # Flat attention weights, WITHOUT bypassing the module. Replacing attn_out by kv.mean(1)
            # would also skip W_v and W_out and so measure something else entirely. With uniform
            # weights every head returns the same token-average of the projected values, so the
            # concatenation is v.mean(1) over the full width and the output projection still applies.
            A = model.attention
            d3 = A.in_proj_weight.shape[0] // 3
            v = kv @ A.in_proj_weight[2 * d3:].T + A.in_proj_bias[2 * d3:]
            attn_out = A.out_proj(v.mean(1))
        elif mode == 'noattn':
            # Sanity floor, not a real hypothesis: z^1 is a learned constant, so with attn_out = 0
            # the model never sees the image at all and MUST fall to chance. If it does not, the
            # ablation machinery is wrong.
            attn_out = torch.zeros_like(attn_out)
        w = attn_w.squeeze(1); diag['L'] = w.shape[-1]           # (B, L): true token-bank size
        diag['w_max'].append(float(w.max(-1).values.mean()))
        diag['w_ent'].append(float((-(w * (w + 1e-12).log()).sum(-1)).mean()))
        diag['a_norm'].append(float(attn_out.norm(dim=-1).mean()))
        diag['z_norm'].append(float(activated_state.norm(dim=-1).mean()))
        state = model.synapses(torch.cat((attn_out, activated_state), dim=-1))
        state_trace = torch.cat((state_trace[:, :, 1:], state.unsqueeze(-1)), dim=-1)
        activated_state = model.trace_processor(state_trace)

        sync_o, da_o, db_o = model.compute_synchronisation(activated_state, da_o, db_o, r_o,
                                                           synch_type='out')
        predictions[..., stepi] = model.output_projector(sync_o)
        certainties[..., stepi] = model.compute_certainty(predictions[..., stepi])
    return predictions, certainties, {k: np.array(v) for k, v in diag.items()}


# Guard: 'intact' must reproduce the model's own forward pass to machine precision, otherwise the
# ablation would be measuring this reimplementation rather than the model.
_m = MODELS[DATASETS[0]]['real'][0]
_x, _ = next(iter(loaders[DATASETS[0]]['test']))
_x = _x[:8].to(device)
with torch.no_grad():
    _pa, _ca, *_ = _m(_x, track=True)
    _pb, _cb, _dg = rollout(_m, _x, mode='intact')
_dp, _dc = float((_pa - _pb).abs().max()), float((_ca - _cb).abs().max())
print(f'rollout vs model.forward: max |dlogit| = {_dp:.2e}, max |dcertainty| = {_dc:.2e}')
assert _dp < 1e-4 and _dc < 1e-4, 'rollout does not reproduce the forward pass'

N_ABL = 6                                   # batches per seed
MODES = ['intact', 'frozen', 'shuffled', 'uniform', 'noattn']

ABLATE = {}
for name in DATASETS:
    per_seed = {m: [] for m in MODES}
    ent_curve = {m: [] for m in MODES}
    for sd in range(N_SEEDS):
        model = MODELS[name]['real'][sd]
        gen = torch.Generator(device=device).manual_seed(sd)
        diags = []
        for m in MODES:
            correct = total = 0
            ents = []
            it = iter(loaders[name]['test'])
            for _ in range(N_ABL):
                try:
                    xb, yb = next(it)
                except StopIteration:
                    break
                xb, yb = xb.to(device), yb.to(device)
                preds, certs, dg = rollout(model, xb, mode=m, gen=gen)
                if m == 'intact':
                    diags.append(dg)
                pred, _ = decision_prediction(preds.cpu(), certs.cpu())
                correct += int((pred == yb.cpu()).sum()); total += len(yb)
                ents.append(certs[:, 0, :].cpu().numpy())
            per_seed[m].append(correct / max(total, 1))
            ent_curve[m].append(np.concatenate(ents, 0).mean(0))
    ABLATE[name] = dict(acc={m: np.array(per_seed[m]) for m in MODES},
                        ent={m: np.stack(ent_curve[m]) for m in MODES},
                        diag={k: np.stack([d[k] for d in diags]).mean(0) for k in diags[0]})
    d = ABLATE[name]['diag']
    L_tok = int(np.max(d['L']))                              # bank size from the attention tensor, not a proxy
    print(f'{name.upper()}  why the query may not matter:')
    print(f'   attention weights: max {d["w_max"].mean():.4f} (uniform would be {1/L_tok:.4f}), '
          f'entropy {d["w_ent"].mean():.3f} of ln(L)={np.log(L_tok):.3f}')
    print(f'   normalised entropy H/ln L = {d["w_ent"].mean() / np.log(L_tok):.3f}   '
          f'effective tokens exp(H) = {np.exp(d["w_ent"].mean()):.1f} of L = {L_tok}')
    print(f'   ||attn_out|| {d["a_norm"].mean():.3f}   vs   ||z|| {d["z_norm"].mean():.3f}   '
          f'ratio {d["a_norm"].mean() / max(d["z_norm"].mean(), 1e-9):.3f}')

    a = ABLATE[name]['acc']
    print(f'{name.upper()}  accuracy at t_dec, mean +- sd over {N_SEEDS} seed(s), chance 0.10')
    for m in MODES:
        drop = 100 * (a['intact'].mean() - a[m].mean()) / max(a['intact'].mean(), 1e-12)
        tag = '' if m == 'intact' else f'   ({-drop:+.1f}% relative to intact)'
        print(f'   {m:9s} {a[m].mean():.4f} +- {a[m].std():.4f}{tag}')
    e = ABLATE[name]['ent']
    print(f'   final entropy: ' + '   '.join(f'{m} {e[m].mean(0)[-1]:.3f}' for m in MODES))
    print()

print('Read: if frozen tracks intact, the query after tick 1 carries little. If shuffled also tracks')
print('intact, the query carries little about THIS image at all, and attention is close to a fixed')
print('read. A gap between frozen and shuffled separates "no adaptation" from "wrong adaptation".')

### [8.4] The state's step against the entropy gradient

$\cos(\Delta z^t,-\nabla_z H)$ at every tick, 64 test images per seed, 3 seeds, against a random direction: its peak, the tick of the peak, and the tick at which the alignment reverses for good (the crossing).

In [ ]:
# [8.4] §8 · does the state's step follow the entropy gradient? -> ALIGN
def alignment_one_seed(name, seed):
    model = MODELS[name]['real'][seed]
    xb, _ = next(iter(loaders[name]['test']))
    xb = xb[:N_GRAD].to(device)
    real, null = [], []
    for t in ALIGN_TICKS:
        frozen_prev = run_to_tick(model, xb, t - 1)     # context defining H_t
        frozen_t    = run_to_tick(model, xb, t)         # context for the actual step
        z_t = frozen_t['activated_state'].clone().detach().requires_grad_(True)
        H = entropy_of_z(model, z_t, frozen_prev).sum()  # sum -> per-sample gradients stay independent
        grad_H, = torch.autograd.grad(H, z_t)
        with torch.no_grad():
            dz = one_tick_step(model, z_t, frozen_t) - z_t
        real.append(F.cosine_similarity(dz, -grad_H, dim=-1).mean().item())
        rng = np.random.default_rng(10_000 * seed + t)
        rd = torch.tensor(rng.normal(size=dz.shape), dtype=torch.float32, device=device)
        rd = rd / rd.norm(dim=-1, keepdim=True)
        null.append(F.cosine_similarity(dz, rd, dim=-1).mean().item())
    return np.array(real), np.array(null)


ALIGN = {}
for name in DATASETS:
    per_seed = [alignment_one_seed(name, s) for s in range(N_SEEDS)]
    real = np.stack([p[0] for p in per_seed]); null = np.stack([p[1] for p in per_seed])
    ALIGN[name] = dict(real=real, null=null)
    rm = real.mean(0)
    RESULTS[f'{name}/align_early'] = mean_sd(real[:, :3].mean(1))
    RESULTS[f'{name}/align_final'] = mean_sd(real[:, -1])
    RESULTS[f'{name}/align_cross'] = (zero_crossing(rm, ALIGN_TICKS) or float('nan'), 0.0)
    print(f'{name.upper()}  cos(dz, -grad H): {[round(v,2) for v in rm]}')
    print(f'    peak {rm.max():+.2f} at tick {ALIGN_TICKS[int(rm.argmax())]} | ticks 1-3 mean {rm[:3].mean():+.2f} | final {rm[-1]:+.2f} | '
          f'zero-crossing tick {zero_crossing(rm, ALIGN_TICKS)} | null ~ {null.mean():+.3f}\n')

### [8.5] The learned decays keep integrating

Retention of the earliest contributions at each sample's decision tick, $\mathbb{E}_{\text{sample}}[\exp(-\delta_{ij}\,t_{\mathrm{dec}})]$, per tracked pair (median 0.90 on MNIST, 0.63 on CIFAR in the paper): the memory keeps integrating rather than saturating.

In [ ]:
# [8.5] §8 · retention of the earliest contributions at the decision tick
# The quantity of interest is the one the CTM authors report: how many pairs have NO meaningful decay
# at all. The natural scale-free measure is RETENTION -- the weight a pair still gives to its very
# first contribution at the end of the thought:
#
#     retention_ij = exp(-delta_ij * H)      in [0, 1]
#
# retention = 1 means the pair never forgets anything (a plain running sum, exactly Eq. 2);
# retention = 0 means its earliest evidence is gone by the end of the window. Unlike a half-life
# threshold, this does not treat "barely halves by the last tick" and "halves every other tick" as
# the same event, and it is directly comparable across pairs and datasets.
#
# HORIZON. The horizon is each SAMPLE'S OWN decision tick, not the full budget T and not the median
# t_dec either. Every other t_dec-dependent quantity in this paper is computed over [1, t_dec], and
# what matters is how much of its history a pair still carries at the moment the model commits --
# which happens at a different tick for every input. So retention is evaluated per (pair, sample)
# and averaged over samples:
#
#     retention_ij = E_sample[ exp(-delta_ij * t_dec(sample)) ]
#
# Using the median horizon instead would bias this DOWNWARD: exp(-delta*t) is convex in t, so by
# Jensen's inequality E[exp(-delta*t)] >= exp(-delta*E[t]), with the gap growing as t_dec spreads
# out. On MNIST the spread is small (median t_dec = T = 20); on CIFAR-10 it is not.
#
# Threshold calibration (at a horizon of 20 ticks): retention >= 0.99 <=> half-life >= 69x the
#                       thought;  retention >= 0.95 <=> half-life >= 14x;  >= 0.90 <=> >= 7x.

RETENTION_LEVELS = (0.99, 0.95, 0.90)   # "no decay at all" / "close to none", at 3 strictnesses

@torch.no_grad()
def tdec_samples(model, loader, n_batches=3):
    """Per-sample decision tick, 1-based. One batched forward pass per batch."""
    ts, it = [], iter(loader)
    for _ in range(n_batches):
        try:
            x, _ = next(it)
        except StopIteration:
            break
        preds, certs, _, _, _, _ = model(x.to(device), track=True)
        ts.append(get_decision_tick(preds.cpu(), certs.cpu()))
    return torch.cat(ts).numpy() + 1


def decay_report(model, horizons):
    """horizons: per-sample decision ticks (1-based). Retention is evaluated at each sample's own
    horizon and averaged over samples, giving one number per neuron pair."""
    out = {}
    for side in ('out', 'action'):
        delta = getattr(model, f'decay_params_{side}').detach().cpu().numpy()
        # delta is clamped to [0,15] by a FORWARD pre-hook, so a final optimizer step can leave it
        # marginally negative; delta <= 0 means r = exp(-delta) >= 1, i.e. no decay at all.
        d = np.clip(delta, 0, None)[None, :]                  # (1, n_pairs)
        retention = np.exp(-d * np.asarray(horizons)[:, None]).mean(0)   # (n_pairs,)
        out[side] = dict(n_pairs=len(delta), delta=np.clip(delta, 0, None), retention=retention,
                         n_at_least={lvl: int((retention >= lvl).sum()) for lvl in RETENTION_LEVELS},
                         median_ret=float(np.median(retention)))
    return out

print('How many neuron pairs effectively do NOT decay over the window the decision is made in?')
print("retention = mean over samples of exp(-delta * t_dec), i.e. the fraction of the first tick's")
print('contribution still counted at the moment that sample was most certain.')
print('retention ~ 1  =>  the pair is a plain running sum, i.e. exactly the undecayed S_t of Eq. (2).\n')
DECAY_H, DECAY_DELTA = {}, {}
for name in DATASETS:
    hz = [tdec_samples(MODELS[name]['real'][s], loaders[name]['test']) for s in range(N_SEEDS)]
    DECAY_DELTA[name] = {sd: np.concatenate(
        [decay_report(MODELS[name]['real'][s], hz[s])[sd]['delta'] for s in range(N_SEEDS)])
        for sd in ('out', 'action')}
    DECAY_H[name] = float(np.median(np.concatenate(hz)))
    _all = np.concatenate(hz)
    print(f'  {name.upper():6s} t_dec over samples: median {np.median(_all):.0f}, '
          f'mean {_all.mean():.1f}, 10-90 pct {np.percentile(_all, 10):.0f}-'
          f'{np.percentile(_all, 90):.0f}  / {T}')
    for side in ('out', 'action'):
        reps = [decay_report(MODELS[name]['real'][s], hz[s])[side] for s in range(N_SEEDS)]
        npair = reps[0]['n_pairs']
        counts = {lvl: np.mean([r['n_at_least'][lvl] for r in reps]) for lvl in RETENTION_LEVELS}
        med = np.mean([r['median_ret'] for r in reps])
        RESULTS[f'{name}/no_decay_{side}'] = (
            counts[0.95], float(np.std([r['n_at_least'][0.95] for r in reps])))
        frac = ' | '.join(f'ret>={lvl:.2f}: {counts[lvl]:5.1f}/{npair} ({counts[lvl]/npair:4.0%})'
                          for lvl in RETENTION_LEVELS)
        print(f'  {name.upper():6s} {side:6s}-side   {frac}   | median retention {med:.2f}')
    print()

print('Read: a high count means the learned accumulator is, within one thought, indistinguishable')
print('from the plain running sum that the rest of the paper analyzes.')

### [8.6] Table 2 · MNIST and CIFAR rows

Collected from Part 2 (accuracy), [8.1] (spectrum) and [8.3] (attention, ablations); mean over seeds. The ImageNet row is in the ImageNet notebook, cell [8.1].

In [ ]:
# [8.6] Table 2 · the MNIST and CIFAR rows (reads RESULTS_ACC, SPECTRA, ABLATE)
LABEL = {'mnist': 'MNIST (D=128)', 'cifar': f'CIFAR ({CIFAR_BACKBONE})'}
print(f"{'model':22s} {'acc.':>6s} {'rho (med)':>10s} {'sigma (med/max)':>16s} {'attn.':>6s} {'frozen':>7s} {'shuffled':>9s}")
for name in DATASETS:
    R, S_ = SPECTRA[name]
    a = ABLATE[name]['acc']; dg = ABLATE[name]['diag']; L_tok = int(np.max(dg['L']))
    rel = lambda m: 100 * (a[m].mean() - a['intact'].mean()) / a['intact'].mean()
    print(f"{LABEL[name]:22s} {RESULTS_ACC[name][0]:6.2f} {np.median(R):10.3f} "
          f"{np.median(S_):7.2f}/{S_.max():<8.2f} {dg['w_ent'].mean() / np.log(L_tok):6.2f} "
          f"{rel('frozen'):+6.1f}% {rel('shuffled'):+8.1f}%")